# Convergence of the normalised OM dissimilarity

Numerical illustration of Proposition *Convergence of the normalised OM dissimilarity*:
$\hat\gamma_n = d_{\mathrm{OM}}(X_{1:n}, Y_{1:n})/n \longrightarrow \gamma(P,Q)$ almost surely, with
a limit that does not depend on the initial laws.

**Design.**
* $\Sigma = \{0,\dots,d-1\}$ with $d = 5$; first-order chains whose transition rows are drawn from a
  Dirichlet($\alpha$) prior with $\alpha = 0.25$. **First draw, no selection of the pair $(P,Q)$.**
* Two configurations: *within* ($X, Y$ independent with the same kernel $P$) and *between*
  ($X \sim P$, $Y \sim Q$).
* **Every replicate starts from its own initial laws**, drawn independently: the two chains of a
  replicate start from the Dirac laws at two states drawn uniformly on $\Sigma$, so no two
  replicates share an initial condition. This is the general setting of the proposition, which
  allows arbitrary initial laws; Dirac laws are the informative case, since any initial law is a
  mixture of them, and also the least favourable one, a diffuse initial law being already closer to
  the stationary law. The spread of the curves at small $n$ therefore includes the transient due to
  the initial condition, and its disappearance is what the second step of the proof asserts.
* $R$ replicates; $\hat\gamma_n$ is evaluated on a logarithmic grid of **nested** prefixes of the
  same trajectories, so each thin curve is a genuine sample path of $\hat\gamma_n$ rather than a
  sequence of independent draws.
* Costs are **fixed in advance**, hence deterministic as the theory requires: the constant scheme
  ($c_{\mathrm{sub}} \equiv 2$, $\delta \equiv 1$) for the main figure; TRATE (estimated on a pilot
  sample **independent** of the sequences entering $\hat\gamma_n$) and random costs for the appendix.
* The two computable bounds $W_{\bar d}(\pi_P,\pi_Q) \le \gamma(P,Q) \le \pi_P^\top S \pi_Q$ are
  superimposed.

Trajectories are simulated **once** and reused across the three cost schemes, so that the three
figures are directly comparable. Shared utilities live in `om_lib.py`; hypothesis checking is in
`assumptions.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import matplotlib.pyplot as plt

from om_lib import (check_assumption_metric, cost_scheme, gamma_hat_paths, product_upper_bound,
                    sample_chain_order1, sample_markov_model, spectral_gap,
                    stationary_distribution_markov, wasserstein_lower_bound)

## Setup: chains and trajectories

In [ ]:
D_STATES  = 5        # alphabet size
ALPHA_DIR = 0.25     # Dirichlet concentration of the transition rows
N_MAX     = 10_000   # largest horizon
N_GRID    = 24       # points of the logarithmic grid
R_REP     = 30       # replicates per configuration
SEED      = 20260802

rng = np.random.default_rng(SEED)

P = sample_markov_model(D_STATES, 1, ALPHA_DIR, rng)["transitions"]
Q = sample_markov_model(D_STATES, 1, ALPHA_DIR, rng)["transitions"]
pi_P = stationary_distribution_markov(P)
pi_Q = stationary_distribution_markov(Q)

grid = np.unique(np.round(np.geomspace(10, N_MAX, N_GRID)).astype(np.int64))


def draw(kernel):
    """R_REP trajectories of `kernel`, each from its own initial law.

    Replicate r starts from the Dirac law at a state drawn uniformly on Sigma, so no two
    replicates share an initial condition. This is the general setting of Proposition 2.1,
    which allows arbitrary initial laws: since any initial law is a mixture of Diracs, the
    extreme points are the informative case, and they are also the least favourable one --
    a diffuse initial law would already be closer to the stationary law.
    """
    return np.stack([sample_chain_order1(kernel, N_MAX, rng, init=int(a))
                     for a in rng.integers(D_STATES, size=R_REP)])


t0 = time.time()
traj = {
    "within":  (draw(P), draw(P)),
    "between": (draw(P), draw(Q)),
}
# pilot sample, independent of the trajectories above, used to estimate the TRATE costs
pilot_sequences = [sample_chain_order1(P, 5000, rng), sample_chain_order1(Q, 5000, rng)]

print(f"simulation: {time.time() - t0:.1f}s")
print("pi_P =", np.round(pi_P, 3), " spectral gap of P =", round(spectral_gap(P), 3))
print("pi_Q =", np.round(pi_Q, 3), " spectral gap of Q =", round(spectral_gap(Q), 3))
print("grid =", grid)

## Sample paths of $\hat\gamma_n$

In [ ]:
COL_IN, COL_OUT = "#1b6ca8", "#c0392b"


def run_cost_scheme(name, S, delta, verbose=True):
    """Check Assumption 1, then compute the sample paths and the bounds for this scheme."""
    checks = check_assumption_metric(S, delta)
    if verbose:
        print(f"[{name}] Assumption 1:")
        for k, v in checks.items():
            print(f"    {k:30s} {v}")
    t0 = time.time()
    paths = {k: gamma_hat_paths(A, B, grid, S, delta) for k, (A, B) in traj.items()}
    bounds = {
        "within":  (wasserstein_lower_bound(pi_P, pi_P, S), product_upper_bound(pi_P, pi_P, S)),
        "between": (wasserstein_lower_bound(pi_P, pi_Q, S), product_upper_bound(pi_P, pi_Q, S)),
    }
    if verbose:
        print(f"[{name}] OM computations: {time.time() - t0:.1f}s")
        for key in traj:
            lo, hi = bounds[key.split("_")[0]]
            print(f"    {key:20s} LB={lo:.3f}   hat-gamma_{grid[-1]}="
                  f"{paths[key][:, -1].mean():.3f}   UB={hi:.3f}")
    return {"paths": paths, "bounds": bounds, "checks": checks, "S": S, "delta": delta}


def plot_gamma_convergence(res, title, filename=None):
    """Single-panel figure: sample paths of hat-gamma_n, with the bounds of Proposition 2.6.

    The legend sits below the axes: the replicate clouds fill the panel at small n, so any
    in-axes placement would hide part of them.
    """
    style = {
        "figure.dpi": 130, "savefig.dpi": 300, "font.family": "serif", "font.size": 10,
        "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8,
        "xtick.labelsize": 9, "ytick.labelsize": 9, "axes.grid": True, "grid.alpha": 0.25,
        "grid.linewidth": 0.5, "axes.spines.top": False, "axes.spines.right": False,
        "lines.linewidth": 1.4, "legend.frameon": False,
    }
    paths, bounds = res["paths"], res["bounds"]
    with plt.rc_context(style):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        for key, col, lab in (("within", COL_IN, r"$(P,P)$"), ("between", COL_OUT, r"$(P,Q)$")):
            ax.plot(grid, paths[key].T, color=col, lw=0.6, alpha=0.16)
            ax.plot(grid, paths[key].mean(0), color=col, lw=1.9, label=rf"$\hat\gamma_n$, {lab}")
        _, hi_in = bounds["within"]
        lo_out, hi_out = bounds["between"]
        ax.axhline(hi_in, color=COL_IN, ls=(0, (5, 3)), lw=1.0, label=r"$\pi_P^{T} S\,\pi_P$")
        ax.axhline(lo_out, color=COL_OUT, ls=(0, (1, 2)), lw=1.2,
                   label=r"$W_{\bar{d}}(\pi_P,\pi_Q)$")
        ax.axhline(hi_out, color=COL_OUT, ls=(0, (5, 3)), lw=1.0, label=r"$\pi_P^{T} S\,\pi_Q$")

        ax.set_xscale("log")
        ax.set_xlabel(r"$n$")
        ax.set_ylabel(r"$\hat\gamma_n = d_{\mathrm{OM}}(X_{1:n},Y_{1:n})\,/\,n$")
        ax.set_ylim(bottom=0.0)
        ax.set_title(title, fontsize=10)
        # three curve entries in the left column, the three bounds in the right one
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
                  handlelength=2.6, columnspacing=1.8, borderaxespad=0.0)
        fig.subplots_adjust(left=0.155, right=0.98, top=0.92, bottom=0.33)
        if filename is not None:
            for ext in ("pdf", "png"):
                fig.savefig(f"Figures/Convergence/{filename}.{ext}", bbox_inches="tight")
            print("figure written to", f"Figures/Convergence/{filename}.pdf")
        plt.show()
    return fig

### Main figure: constant costs

In [ ]:
S_const, delta_const = cost_scheme("constant", D_STATES, sub=2.0, indel=1.0)
res_const = run_cost_scheme("constant", S_const, delta_const)
_ = plot_gamma_convergence(
    res_const,
    title="constant costs: $c_{\\mathrm{sub}}\\equiv 2$, $\\delta\\equiv 1$",
    filename="gamma_convergence_constant",
)

### Appendix: TRATE and random costs, same trajectories

In [ ]:
S_trate, delta_trate = cost_scheme("trate", D_STATES, pilot_sequences=pilot_sequences, indel=1.0)
res_trate = run_cost_scheme("trate", S_trate, delta_trate)
_ = plot_gamma_convergence(
    res_trate,
    title="TRATE costs: $c_{\\mathrm{sub}}=2-\\hat{P}-\\hat{P}^{T}$, $\\delta\\equiv 1$",
    filename="gamma_convergence_trate",
)
print("\nTRATE cost matrix:\n", np.round(S_trate, 3))

In [ ]:
S_rand, delta_rand = cost_scheme("random", D_STATES, rng=np.random.default_rng(SEED + 1),
                                 low=1.2, high=2.0, indel=1.0)
res_rand = run_cost_scheme("random", S_rand, delta_rand)
_ = plot_gamma_convergence(
    res_rand,
    title="random costs: $c_{\\mathrm{sub}}\\sim\\mathcal{U}[1.2,2]$, $\\delta\\equiv 1$",
    filename="gamma_convergence_random",
)